<a href="https://colab.research.google.com/github/MithunSrinivas28/wafer-defect-ai/blob/addfeature/Wafer_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Mount Google Drive**

In [5]:
import tensorflow as tf
tf.keras.backend.clear_session()


In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Load Dataset Using TensorFlow

In [21]:
TRAIN_PATH = "/content/drive/MyDrive/Datasets/train"
TEST_PATH  = "/content/drive/MyDrive/Datasets/test"


In [8]:
#import os
#import shutil
#import random
#import math

# -------- CHANGE ONLY IF NEEDED --------
#TRAIN_DIR = "/content/drive/MyDrive/Datasets/train"
T#EST_DIR  = "/content/drive/MyDrive/Datasets/test"

# --------------------------------------

#os.makedirs(TEST_DIR, exist_ok=True)

#for class_name in os.listdir(TRAIN_DIR):

    #train_class_path = os.path.join(TRAIN_DIR, class_name)
    #test_class_path = os.path.join(TEST_DIR, class_name)

    #if not os.path.isdir(train_class_path):
        #continue

    #os.makedirs(test_class_path, exist_ok=True)

    #images = [img for img in os.listdir(train_class_path)
              #if img.lower().endswith((".jpg", ".jpeg", ".png"))]

    #total = len(images)
    #test_count = math.ceil(0.20 * total)   # 20% for test
    #train_count = total - test_count       # remaining 80%

    #print(f"\nClass: {class_name}")
    #print(f"Total images: {total}")
    #print(f"Train target: {train_count}")
    #print(f"Test target:  {test_count}")

    #selected_for_test = random.sample(images, test_count)

    #for img in selected_for_test:
        #src = os.path.join(train_class_path, img)
        #dst = os.path.join(test_class_path, img)
        #shutil.move(src, dst)

    #print(f"✅ Moved {test_count} images to test/{class_name}")

#print("\n🎯 80–20 split completed successfully!")


Class: bridge
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/bridge

Class: clean
Total images: 165
Train target: 132
Test target:  33
✅ Moved 33 images to test/clean

Class: cmp
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/cmp

Class: crack
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/crack

Class: ler
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/ler

Class: open
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/open

Class: vias
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/vias

Class: others
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/others

🎯 80–20 split completed successfully!


In [22]:
train_data = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=(224,224),
    batch_size=32,
    shuffle=True
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_PATH,
    image_size=(224,224),
    batch_size=32,
    shuffle=False
)


Found 1032 files belonging to 8 classes.
Found 264 files belonging to 8 classes.


In [10]:
import tensorflow as tf

# Reload only to get class names
temp_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/drive/MyDrive/Datasets/train",
    image_size=(224,224),
    batch_size=32
)

class_names = temp_ds.class_names
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Num classes:", NUM_CLASSES)


Found 1032 files belonging to 8 classes.
Classes: ['bridge', 'clean', 'cmp', 'crack', 'ler', 'open', 'others', 'vias']
Num classes: 8


In [11]:
import os

for c in os.listdir(TRAIN_PATH):
    print(c, len(os.listdir(TRAIN_PATH + "/" + c)))


bridge 129
clean 129
cmp 129
crack 129
ler 129
open 129
vias 129
others 129


### Normalize + Add Data Augmentation

In [12]:
from tensorflow.keras import layers

# Normalize (0–255 → 0–1)
normalization = layers.Rescaling(1./255)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
])



# Apply to datasets
train_data = train_data.map(lambda x, y: (normalization(data_augmentation(x)), y))
test_data  = test_data.map(lambda x, y: (normalization(x), y))


### Build the MobileNet Model (Your AI Brain)

In [13]:

import tensorflow as tf
from tensorflow.keras import layers, models

base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224,224,3),
    include_top=False,
    #weights="imagenet"   # ✅ IMPORTANT
)

# Freeze backbone
base_model.trainable = True  # ✅ IMPORTANT

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()



4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 576)            │         2,304 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,016,312 (3.88 MB)

 Trainable params: 1,003,048 (3.83 MB)

 Non-trainable params: 13,264 (51.81 KB)

##** Compile the Model**

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]   # ONLY accuracy
)


In [15]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

y = []
for _, labels in train_data:
    y.extend(labels.numpy())

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


{0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0), 6: np.float64(1.0), 7: np.float64(1.0)}


### Train the Model

In [ ]:
EPOCHS = 15   # good for small dataset

history = model.fit(
    train_data,
    validation_data=train_data,
    epochs=15,
    class_weight=class_weights
)




Epoch 1/15


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.

In [ ]:
model.save("/content/drive/MyDrive/Wafer-detect.keras")



setup

In [16]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

from tensorflow.keras import layers, models


Define Dataset Paths

In [24]:
TRAIN_PATH = "/content/drive/MyDrive/Datasets/train"
TEST_PATH  = "/content/drive/MyDrive/Datasets/test"


Load Dataset

In [44]:
train_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=(224,224),
    color_mode="grayscale",
    batch_size=32,
    shuffle=True
)

test_data = tf.keras.utils.image_dataset_from_directory(
    TEST_PATH,
    image_size=(224,224),
    color_mode="grayscale",
    batch_size=32,
    shuffle=True
)



Found 1032 files belonging to 8 classes.
Found 264 files belonging to 8 classes.


Get Class Names

In [26]:
class_names = train_data.class_names
print(class_names)


['bridge', 'clean', 'cmp', 'crack', 'ler', 'open', 'others', 'vias']


Normalize Images (Preprocessing)

In [45]:
norm = tf.keras.layers.Rescaling(1./255)

def gray_to_rgb(x):
    return tf.repeat(x, repeats=3, axis=-1)   # 1 → 3 channels

train_data = train_data.map(lambda x,y: (gray_to_rgb(norm(x)), y))
test_data  = test_data.map(lambda x,y: (gray_to_rgb(norm(x)), y))



In [46]:
for images, labels in train_data.take(1):
    print(images.shape)


(32, 224, 224, 3)


Data Augmentation (Prevent Overfitting)

In [28]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])


Prepare Final Dataset Pipeline

In [29]:
train_data = train_data.map(
    lambda x, y: (data_augmentation(normalization(x)), y)
)

test_data = test_data.map(
    lambda x, y: (normalization(x), y)
)


Load Pretrained Model (Transfer Learning)

In [30]:
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)


Freeze Backbone (Stage 1 Training)

In [31]:
base_model.trainable = False


Build Your Custom Model

In [32]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation="softmax")
])



Compile Model

In [38]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


Train again:

In [40]:
norm = tf.keras.layers.Rescaling(1./255)

def gray_to_rgb(x):
    return tf.repeat(x, repeats=3, axis=-1)

train_data = train_data.map(
    lambda x,y: (gray_to_rgb(norm(x)), y)
)

test_data = test_data.map(
    lambda x,y: (gray_to_rgb(norm(x)), y)
)


STAGE‑2 FINE‑TUNING

In [59]:
base_model.trainable = False


In [60]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),   # back to normal
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [61]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=12
)


Epoch 1/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 30s 735ms/step - accuracy: 0.2583 - loss: 1.7538 - val_accuracy: 0.3371 - val_loss: 1.5128
Epoch 2/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 22s 664ms/step - accuracy: 0.3098 - loss: 1.5870 - val_accuracy: 0.4545 - val_loss: 1.4297
Epoch 3/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 22s 652ms/step - accuracy: 0.3939 - loss: 1.5174 - val_accuracy: 0.4432 - val_loss: 1.4087
Epoch 4/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 22s 657ms/step - accuracy: 0.4062 - loss: 1.4981 - val_accuracy: 0.4545 - val_loss: 1.3973
Epoch 5/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 22s 664ms/step - accuracy: 0.4382 - loss: 1.4594 - val_accuracy: 0.4659 - val_loss: 1.3885
Epoch 6/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 41s 675ms/step - accuracy: 0.3889 - loss: 1.4858 - val_accuracy: 0.4697 - val_loss: 1.3821
Epoch 7/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 22s 670ms/step - accuracy: 0.4355 - loss: 1.4573 - val_accuracy: 0.4583 - val_loss: 1.3790
Epoch 8/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 22s 663ms/step - accuracy: 0.4183 - loss: 1.4570 - val_accu

In [55]:
from sklearn.utils import class_weight
import numpy as np

labels = np.concatenate([y for x,y in train_data], axis=0)
cw = class_weight.compute_class_weight(
    "balanced",
    classes=np.unique(labels),
    y=labels
)
class_weights = dict(enumerate(cw))


In [57]:
base_model.trainable = True
for layer in base_model.layers[:-30]:   # unfreeze only last 15
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)



In [58]:
model.fit(train_data,
          validation_data=test_data,
          epochs=8,
          class_weight=class_weights)


Epoch 1/8
33/33 ━━━━━━━━━━━━━━━━━━━━ 34s 750ms/step - accuracy: 0.2547 - loss: 2.0195 - val_accuracy: 0.3788 - val_loss: 1.4572
Epoch 2/8
33/33 ━━━━━━━━━━━━━━━━━━━━ 25s 754ms/step - accuracy: 0.2743 - loss: 1.6736 - val_accuracy: 0.3447 - val_loss: 1.5076
Epoch 3/8
33/33 ━━━━━━━━━━━━━━━━━━━━ 24s 728ms/step - accuracy: 0.3218 - loss: 1.5929 - val_accuracy: 0.2917 - val_loss: 1.5483
Epoch 4/8
33/33 ━━━━━━━━━━━━━━━━━━━━ 23s 676ms/step - accuracy: 0.3174 - loss: 1.5260 - val_accuracy: 0.3220 - val_loss: 1.5799
Epoch 5/8
33/33 ━━━━━━━━━━━━━━━━━━━━ 24s 729ms/step - accuracy: 0.3587 - loss: 1.5104 - val_accuracy: 0.3295 - val_loss: 1.6106
Epoch 6/8
33/33 ━━━━━━━━━━━━━━━━━━━━ 41s 731ms/step - accuracy: 0.3700 - loss: 1.4797 - val_accuracy: 0.3220 - val_loss: 1.6445
Epoch 7/8
33/33 ━━━━━━━━━━━━━━━━━━━━ 24s 717ms/step - accuracy: 0.3667 - loss: 1.4847 - val_accuracy: 0.3030 - val_loss: 1.6822
Epoch 8/8
33/33 ━━━━━━━━━━━━━━━━━━━━ 22s 666ms/step - accuracy: 0.3967 - loss: 1.4271 - val_accuracy: 0.

In [62]:
for layer in model.layers[0].layers:
    if "conv" in layer.name.lower():
        print(layer.name)


conv
conv_bn
expanded_conv_depthwise_pad
expanded_conv_depthwise
expanded_conv_depthwise_bn
expanded_conv_squeeze_excite_avg_pool
expanded_conv_squeeze_excite_conv
expanded_conv_squeeze_excite_relu
expanded_conv_squeeze_excite_conv_1
expanded_conv_squeeze_excite_mul
expanded_conv_project
expanded_conv_project_bn
expanded_conv_1_expand
expanded_conv_1_expand_bn
expanded_conv_1_depthwise_pad
expanded_conv_1_depthwise
expanded_conv_1_depthwise_bn
expanded_conv_1_project
expanded_conv_1_project_bn
expanded_conv_2_expand
expanded_conv_2_expand_bn
expanded_conv_2_depthwise
expanded_conv_2_depthwise_bn
expanded_conv_2_project
expanded_conv_2_project_bn
expanded_conv_2_add
expanded_conv_3_expand
expanded_conv_3_expand_bn
expanded_conv_3_depthwise_pad
expanded_conv_3_depthwise
expanded_conv_3_depthwise_bn
expanded_conv_3_squeeze_excite_avg_pool
expanded_conv_3_squeeze_excite_conv
expanded_conv_3_squeeze_excite_relu
expanded_conv_3_squeeze_excite_conv_1
expanded_conv_3_squeeze_excite_mul
expande

In [65]:
last_conv = "conv_1"

In [66]:
model.save("/content/drive/MyDrive/wafer_xai_model.keras")
print("Saved XAI model")


Saved XAI model


In [68]:
print(class_names)


['bridge', 'clean', 'cmp', 'crack', 'ler', 'open', 'others', 'vias']
